# 01 — Ingestion (bronze layer)

**Purpose.** Read the ten raw inputs under `data/raw/`, validate each
against its documented schema, and materialise each as a schema-faithful
parquet snapshot under `data/processed/<source>/<filename>.parquet`. No
transforms, joins, derivations, renames, or type coercions beyond what
`pd.read_csv` / `pd.read_excel` does natively. Documented preprocessing
(AWS skiprows=1, Ember column reconciliation, IM3 layer filtering,
Aqueduct sentinel handling, indicator/weight filters) is deferred to
Phase 2.

**Inputs.**

- `../data/raw/epoch/epoch_all_models.csv`
- `../data/raw/ember/ember_yearly.csv`
- `../data/raw/ember/ember_us_states.csv`
- `../data/raw/im3/im3_open_source_data_center_atlas_v2026_02_09.csv`
- `../data/raw/aqueduct/Aqueduct40_rankings_download_Y2023M07D05.xlsx` (sheets `country_baseline`, `province_baseline`)
- `../data/raw/jegham/DataSnapshotOct26.csv`
- `../data/raw/jegham/artificialanalysis_environmental.csv`
- `../data/raw/jegham/Monthly_LLM_Environmental_Footprint.csv`
- `../data/raw/jegham/AWS_Env_Multipliers.csv`
- `../data/raw/jegham/Microsoft_Env_Multipliers.csv`

**Outputs.**

- `../data/processed/epoch/epoch_all_models.parquet`
- `../data/processed/ember/ember_yearly.parquet`
- `../data/processed/ember/ember_us_states.parquet`
- `../data/processed/im3/im3_open_source_data_center_atlas_v2026_02_09.parquet`
- `../data/processed/aqueduct/Aqueduct40_country_baseline.parquet`
- `../data/processed/aqueduct/Aqueduct40_province_baseline.parquet`
- `../data/processed/jegham/DataSnapshotOct26.parquet`
- `../data/processed/jegham/artificialanalysis_environmental.parquet`
- `../data/processed/jegham/Monthly_LLM_Environmental_Footprint.parquet`
- `../data/processed/jegham/AWS_Env_Multipliers.parquet`
- `../data/processed/jegham/Microsoft_Env_Multipliers.parquet`
- `../data/processed/README.md`

In [5]:
from pathlib import Path

import pandas as pd

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")

for source in ["epoch", "ember", "im3", "aqueduct", "jegham"]:
    (PROCESSED / source).mkdir(parents=True, exist_ok=True)

In [6]:
def validate_and_write(df, source, filename, expected_columns, expected_min_rows):
    missing = [c for c in expected_columns if c not in df.columns]
    assert not missing, f"{source}/{filename}: missing expected columns {missing}"
    assert (
        len(df) >= expected_min_rows
    ), f"{source}/{filename}: {len(df)} rows < expected min {expected_min_rows}"

    extras = [c for c in df.columns if c not in expected_columns]
    print(
        f"{source}/{filename}: rows={len(df)} cols={len(df.columns)} "
        f"extras={len(extras)}"
    )

    out = PROCESSED / source / f"{filename}.parquet"
    df.to_parquet(out, engine="pyarrow", index=False)

## Epoch

Single file: `epoch_all_models.csv`. Default `pd.read_csv`. The
`Training power draw (W)` column has documented partial coverage
(populated for ~481 post-2023 models); validation only checks that the
column is present, not how densely it is populated.

In [7]:
df_epoch = pd.read_csv(RAW / "epoch" / "epoch_all_models.csv")

validate_and_write(
    df_epoch,
    source="epoch",
    filename="epoch_all_models",
    expected_columns=[
        "Model",
        "Publication date",
        "Organization",
        "Training compute (FLOP)",
        "Parameters",
        "Training hardware",
        "Training power draw (W)",
    ],
    expected_min_rows=3000,
)

power_draw_populated = df_epoch["Training power draw (W)"].notna().sum()
print(f"epoch/epoch_all_models: Training power draw (W) populated rows = {power_draw_populated}")

epoch/epoch_all_models: rows=3509 cols=57 extras=50
epoch/epoch_all_models: Training power draw (W) populated rows = 771


## Ember

Two files in long format. Default `pd.read_csv` for both. Per the
Ember PROVENANCE, the yearly file uses `Area` + `ISO 3 code` for
country identification while the US states file uses `State` + `State
code`; reconciliation is a Phase 2 task. The `Variable` column carries
many measures (generation, demand, emissions, intensity, etc.); we do
not filter to `CO2 intensity` here.

In [8]:
df_ember_yearly = pd.read_csv(RAW / "ember" / "ember_yearly.csv")

validate_and_write(
    df_ember_yearly,
    source="ember",
    filename="ember_yearly",
    expected_columns=[
        "Area",
        "ISO 3 code",
        "Year",
        "Variable",
        "Unit",
        "Value",
    ],
    expected_min_rows=200000,
)

ember/ember_yearly: rows=371020 cols=18 extras=12


In [9]:
df_ember_states = pd.read_csv(RAW / "ember" / "ember_us_states.csv")

validate_and_write(
    df_ember_states,
    source="ember",
    filename="ember_us_states",
    expected_columns=[
        "State",
        "State code",
        "Year",
        "Variable",
        "Unit",
        "Value",
    ],
    expected_min_rows=50000,
)

ember/ember_us_states: rows=73442 cols=13 extras=7


## IM3

Single file: the IM3 open-source data center atlas. Default
`pd.read_csv`. The atlas has multiple geographic layers in the `type`
column (campus, building, etc.); we do not filter to `type == "campus"`
here — that is a Phase 2 concern.

In [10]:
df_im3 = pd.read_csv(
    RAW / "im3" / "im3_open_source_data_center_atlas_v2026_02_09.csv"
)

validate_and_write(
    df_im3,
    source="im3",
    filename="im3_open_source_data_center_atlas_v2026_02_09",
    expected_columns=[
        "id",
        "state",
        "state_abb",
        "county",
        "operator",
        "name",
        "sqft",
        "lat",
        "lon",
        "type",
    ],
    expected_min_rows=1000,
)

im3/im3_open_source_data_center_atlas_v2026_02_09: rows=1479 cols=13 extras=3


## Aqueduct

One XLSX, two sheets, two parquet outputs. Default
`pd.read_excel(path, sheet_name=...)` for both sheets — the headers sit
on row 0, and PROVENANCE confirms no preprocessing at ingestion. We do
not filter to `indicator_name == "bws"` or `weight == "Tot"`, and we do
not replace -9999 sentinels (a known Singapore artefact for the `bws`
indicator); both are Phase 2 transformations.

In [11]:
df_aq_country = pd.read_excel(
    RAW / "aqueduct" / "Aqueduct40_rankings_download_Y2023M07D05.xlsx",
    sheet_name="country_baseline",
)

validate_and_write(
    df_aq_country,
    source="aqueduct",
    filename="Aqueduct40_country_baseline",
    expected_columns=[
        "gid_0",
        "name_0",
        "indicator_name",
        "weight",
        "score",
        "score_ranked",
        "cat",
        "label",
        "un_region",
        "wb_region",
    ],
    expected_min_rows=200,
)

aqueduct/Aqueduct40_country_baseline: rows=2456 cols=10 extras=0


In [12]:
df_aq_province = pd.read_excel(
    RAW / "aqueduct" / "Aqueduct40_rankings_download_Y2023M07D05.xlsx",
    sheet_name="province_baseline",
)

validate_and_write(
    df_aq_province,
    source="aqueduct",
    filename="Aqueduct40_province_baseline",
    expected_columns=[
        "name_0",
        "name_1",
        "gid_1",
        "indicator_name",
        "weight",
        "score",
        "cat",
        "label",
    ],
    expected_min_rows=3000,
)

aqueduct/Aqueduct40_province_baseline: rows=42771 cols=12 extras=4
